<a href="https://colab.research.google.com/github/sonawanenavanit/Langchain_HuggingFace_Project/blob/main/RAG_PDF_Reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
# !pip install PyPDF2
# !pip install chromadb
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.3 MB/s eta 0:00:00


In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from PyPDF2 import PdfReader
import chromadb
from groq import Groq

In [12]:
from PyPDF2 import PdfReader

reader = PdfReader("/content/budget_speech.pdf")
number_of_pages = len(reader.pages)
page = reader.pages[0]
text = page.extract_text()
text

'GOVERNMENT OF INDIA\nBUDGET 2025-2026\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2025'

In [17]:
book =''
for page in reader.pages:
  text = page.extract_text()
  book = book + text

In [20]:
len(book.split())

14893

In [26]:

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.create_documents([book])

In [29]:
len(texts)

117

In [34]:
chunks = [d.page_content for d in texts]

In [ ]:
chunks

In [40]:
client = chromadb.Client()
collection = client.create_collection(name = 'collection1')

In [42]:
ids = [str(i) for i in range(len(chunks))]
collection.add(documents=chunks, ids=ids)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 103MiB/s]


In [44]:
query = "What action is taken for cancer medicines"
result = collection.query(query_texts=[query], n_results=3)
context = " ".join(result['documents'][0])

In [45]:
prompt = '''Act as AI assistant for questio answering system.
          All information with respect to question will be provided and question will also be provided.
          oyu find out the exact answer of question in provided information and give proper answer in proper language. If you don't get the answer from the provided information,
          just say "No Information found in Given Context".
          information:{}
          question: {}'''.format(context,query)



In [49]:
print(prompt)

Act as AI assistant for questio answering system. 
          All information with respect to question will be provided and question will also be provided.
          oyu find out the exact answer of question in provided information and give proper answer in proper language. If you don't get the answer from the provided information,
          just say "No Information found in Given Context".
          information:to a cess.  
117. I shall now take up sector specific proposals.  
Relief on import of Drugs/Medicines  
118.  To provide relief to patients, particularly those suffering from cancer, 
rare diseases and other severe chronic diseases, I propose to add 36 lifesaving 
drugs and medicines to the list of medicines fully exempted from Basic Customs 
Duty (BCD).  I also pro pose to add 6 lifesaving medicines to the list attracting 
concessional customs duty of 5%. Full exemption and concessional duty will 
also respectively apply on the bulk drugs for manufacture of the above.  
119. S

In [56]:


client = Groq(
    # This is the default and can be omitted
    api_key='gsk_x7BNQ3qilxOS8E6ic5WGdyb3FYMjcvUJFC93dhjPl6g0fvNtrF',
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": prompt
        }

    ],
    model="llama-3.3-70b-versatile",
)

answer = chat_completion.choices[0].message.content

In [57]:
answer

'To provide relief to patients, particularly those suffering from cancer, the following actions are taken: \n\n1. 36 lifesaving drugs and medicines are added to the list of medicines fully exempted from Basic Customs Duty (BCD).\n2. 6 lifesaving medicines are added to the list attracting concessional customs duty of 5%.\n3. Full exemption and concessional duty will also apply on the bulk drugs for manufacture of the above medicines.\n4. Additionally, Day Care Cancer Centres will be set up in all district hospitals in the next 3 years, with 200 Centres to be established in 2025-26. \n\nThese measures aim to provide relief to cancer patients by making lifesaving medicines more accessible and affordable, and by expanding medical infrastructure for cancer treatment.'